# 01. Data overview and inventory

This notebook inspects all data sources used in the World Education project. It documents:
- **Schema**: column names and types for each dataset
- **Row counts and basic stats**: shape, missing values, sample rows
- **Mapping to logical tables**: which CSVs correspond to the tables used in SQL (economy, population, education, education_quality, continents)

**Data sources**: Our World in Data, World Bank, World Population Review (see project README).

## Setup

Paths are relative to the project root `world_education/`. Run this notebook from the project root, or set `PROJECT_ROOT` below.

In [ ]:
import pandas as pd
from pathlib import Path

# Project root: world_education/ (parent of notebooks/). Run notebook from notebooks/ so ".." is correct.
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"

def csv_path(*parts):
    return DATA_DIR.joinpath(*parts)

## Helper: summarize a DataFrame

Use this to get schema, shape, nulls, and a sample in one go.

In [ ]:
def summarize_df(name, df, sample_n=5):
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"\nColumns and dtypes:")
    print(df.dtypes.to_string())
    print(f"\nMissing values:")
    print(df.isnull().sum().to_string())
    print(f"\nSample rows:")
    display(df.head(sample_n))

## Core tables (used in SQL)

These five CSVs in `data/` are the main inputs for the SQLite database and analysis.

In [ ]:
# 1. Continents — country → continent, subregion lookup
continents = pd.read_csv(csv_path("continents.csv"))
summarize_df("continents (logical: continents)", continents)

In [ ]:
# 2. Economy — GDP per capita, government expenditure on education
economy = pd.read_csv(csv_path("economy.csv"))
summarize_df("economy (logical: economy)", economy)

In [ ]:
# 3. Population — country, population, urban %
population = pd.read_csv(csv_path("population.csv"))
summarize_df("population (logical: population)", population)

In [ ]:
# 4. Education — enrollment and completion ratios (primary, secondary, tertiary)
education = pd.read_csv(csv_path("education.csv"))
summarize_df("education (logical: education)", education)

In [ ]:
# 5. Education quality — PISA scores (reading, mathematics, science)
quality_education = pd.read_csv(csv_path("quality_education.csv"))
summarize_df("quality_education (logical: education_quality)", quality_education)

## Optional: Our World in Data (OWID) CSVs

These live under `data/Economy/` and `data/Education/`. They are time-series and can be used for deeper analysis or to cross-check the aggregated core tables.

In [ ]:
owid_files = [
    "Economy/population.csv",
    "Economy/population-and-demography.csv",
    "Economy/total-government-expenditure-on-education-gdp.csv",
    "Education/primary-completion-rate-of-relevant-age-group.csv",
    "Education/gross-enrollment-ratio-in-primary-education.csv",
    "Education/gross-enrollment-ratio-in-secondary-education.csv",
    "Education/gross-enrollment-ratio-in-tertiary-education.csv",
    "Education/completion-rate-of-lower-secondary-education.csv",
    "Education/share-of-the-population-with-completed-tertiary-education.csv",
    "Education/pisa-test-score-mean-performance-on-the-reading-scale.csv",
    "Education/pisa-test-score-mean-performance-on-the-mathematics-scale.csv",
    "Education/pisa-test-score-mean-performance-on-the-science-scale.csv",
]

for rel_path in owid_files:
    p = csv_path(rel_path)
    if not p.exists():
        print(f"Skip (not found): {rel_path}")
        continue
    df = pd.read_csv(p)
    print(f"{rel_path}: {df.shape[0]} rows × {df.shape[1]} cols — columns: {list(df.columns)[:6]}{'...' if len(df.columns) > 6 else ''}")

## Data dictionary (logical tables)

| Logical table      | CSV file          | Purpose |
|--------------------|-------------------|--------|
| `continents`       | `continents.csv`   | Country → continent, subregion |
| `economy`          | `economy.csv`      | GDP per capita, avg gov expenditure on education (% GDP) |
| `population`       | `population.csv`  | Population (2020), urban % |
| `education`        | `education.csv`   | Avg enrollment & completion % (primary, secondary, tertiary) |
| `education_quality`| `quality_education.csv` | Avg PISA (reading, maths, science) |

**Join key**: `Country` (in economy, population, education, quality_education) and `country` (in continents). Names must be aligned for joins — we will standardize in the cleaning step.